# 11 · Look-up tables without interpolation

Notebook 10's tables do little arithmetic, but they are slow as a graph: reading a 4-D table
between its points is simplex interpolation - sort the four fractions, find five corners,
weigh them - and as ONNX operators that is some sixty full-size tensors written to memory and
read back, per stage. This notebook tries two ways to read **one entry, at an exact integer
address**, with no interpolation, no sort and no weights to work out.

**A binned table** (`S`, `SDY x2`) - the entry is the answer:

    LR luma ──► 4 taps per kernel and rotation ──► centre level + 3 differences, each binned by a small index table
                                                   ──► their sum is the row ──► one read ──► turn back, average ──► HR luma correction

- **The address is the centre and the differences from it.** A table that is never
  interpolated needs a row for every input it can see, and four 8-bit taps are four billion.
  But neighbouring pixels are mostly alike - notebook 10 found 85% of its reads within one
  table step of the diagonal - so the row is the centre's brightness in 8 levels, and each of
  the three other taps' **difference** from it in 19 bins: fine near 0 (0, ±1, ±2, ±3-4,
  ±5-7, ...) and coarse far from it (±61-255). That is 8 x 19³ = 54,872 rows a kernel.
- **The bins are a table too.** A tap's rounded value (or difference) is an index into a small
  table of integers that holds its bin already multiplied by its place in the row, so four
  pixels become a row with one convolution, a cast, one read of the small table and a sum.
- **The rest is notebook 10's**: MuLUT's kernels S, D and Y, each in four rotations sharing one
  table; a second stage on the first one's output.

**A 3x3 pattern filter** (`3x3 pattern`) - the entry is a filter:

    LR luma ──► one 3x3 convolution: the 8 differences from the centre ──► their signs: the pattern ──► one read: its filter
                                                                        └──────────────────────────────────► × ──► HR luma correction

- **Every combination has a row.** The pattern is each of the 8 neighbours darker than the
  centre, the same (within 2 of 255 steps) or brighter: 3⁸ = 6,561 of them, all in the table.
- **The row is a filter**: for each of the four x2 sub-pixels, a weight per neighbour and a
  bias, applied to the actual differences - 36 multiply-adds. Since the filter is linear in
  the pixels, a strong edge gets a proportionally strong correction.

Both are trained as tables: the entries are the parameters - there is no network to fill them
from - and they start at zero, so untrained either model is bicubic. Both correct luma only.

Trained with the shared recipe (`training.fit`) and scored on notebook 0's validation split beside notebook 10's
interpolated tables and notebook 7's model.

Needs: `00_data_preprocess.ipynb`, and the scores of notebooks 1 (bicubic), 4 and 8 (notebook 7's model at three
seeds) and 10.

In [1]:
"""Settings."""

import copy
import os

import numpy as np
import torch
import torch.nn as nn

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, training, webexport
from helpers.training import DEVICE

NOTEBOOK = 11
CENTRE_LEVELS = 8                         # the centre's brightness, in 8 steps of 32
EDGES = (1, 2, 3, 5, 8, 13, 21, 35, 61)   # where a difference's bins start, in 8-bit steps: 0, 1, 2, 3-4, 5-7, ...
DEAD_ZONE = 2.5     # 8-bit steps: a neighbour within 2 of the centre is "the same" in a 3x3 pattern
SEEDS = (0, 1, 2)   # S trained three times: how far luck alone moves it
RUN_DIR = training.CHECKPOINT_DIR / "fastlut_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
trim = training.trimmer(manifest)   # cuts the halo off an output: the border is padding, not picture
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]    # notebook 1's scores of the bar
others = {name: scores for (notebook, name), scores in recorded.items()
          if (notebook in (4, 8) and name.startswith("luma 24 + 2 blocks"))
          or (notebook == 10 and name.endswith("table, fine-tuned"))}   # notebook 7's model, notebook 10's tables

## 1. Model

**The binned table.** Per stage, every kernel in each of four rotations - a *group* - reads four taps: the centre
and three around it.

- **Read.** One frozen convolution writes, per group, the centre and the three differences
  from it in 8-bit steps, plus 0.5 (and 255.5 for a difference) so that dropping the fraction
  rounds and nothing is negative. A cast to integers drops the fraction.
- **Bin.** Each channel adds where its part of the index table starts, and reads it: a
  centre value gives its level times 19³ (plus where its kernel's table starts), a
  difference its bin times 19², 19 or 1.
- **Sum.** The four numbers add up to the row - three integer `Add`s, not a `ReduceSum`, which
  the browser's WebGPU provider has no integer kernel for. Every row is a different combination of
  bins, and every combination has a row.
- **Answer.** One read of the table gives the group's answer: one number in an earlier
  stage (a correction of the LR luma the next stage reads), the four x2 sub-pixels in the
  last. A frozen 1x1 convolution turns each rotation back, averages the rotations and adds
  the kernels, as in notebook 10.

**The 3x3 pattern filter.** One frozen 3x3 convolution writes the 8 differences from the
centre. Scaled so that the dead zone is half a step, rounded and clipped to -1..1, they are
the pattern's 8 base-3 digits; a 1x1 convolution adds them up in two halves of four (each
0..80), and the halves meet in float32, where every row number is exact, before the cast:
  the browser's WebGPU provider has no integer `Mul`. Rounding is `floor(x + 0.5)`, which it
  runs, where `Round` it does not; on 8-bit pixels the two never differ, since no scaled
  difference lands on a half. The row read is the filter,
applied to the differences.

In both, luma is clamped to 0..1 first, so whatever a graph is fed, every address stays
inside its table.

In [2]:
LUMA = torch.tensor(models.LUMA_709)
ROTATIONS = 4
KERNELS = {   # a kernel's four taps, (down, across) from the pixel it answers for: MuLUT's
    "S": ((0, 0), (0, 1), (1, 0), (1, 1)),   # SR-LUT's 2x2 square
    "D": ((0, 0), (0, 2), (2, 0), (2, 2)),   # the square, dilated
    "Y": ((0, 0), (1, 1), (1, 2), (2, 1)),   # the diagonal neighbour and two beside it
}
BINS = 2 * len(EDGES) + 1   # 19: the bins either side of 0, and 0


def rotate(offset, turns):
    """An offset (down, across) turned by `turns` quarter turns."""
    dy, dx = offset
    for _ in range(turns):
        dy, dx = dx, -dy
    return dy, dx


def frozen(conv, weight, bias=None):
    """`conv` with these weights (and bias), never trained."""
    with torch.no_grad():
        conv.weight.copy_(weight.reshape(conv.weight.shape))
        if bias is not None:
            conv.bias.copy_(bias)
    for parameter in conv.parameters():
        parameter.requires_grad_(False)
    return conv


def difference_bin(d):
    """The bin of an 8-bit difference -255..255: 0 for the most negative, BINS // 2 for 0."""
    magnitude = torch.bucketize(d.abs(), torch.tensor(EDGES), right=True)
    return torch.where(d >= 0, magnitude, -magnitude) + len(EDGES)


class FastLut(models.Upscaler):
    """Bicubic colour, and a luma correction read from tables at one exact address each."""

    def __init__(self, kernels="S", stages=1, scale=2):
        super().__init__(scale, skip=5)
        self.kernels, self.stages = kernels, stages
        G = self.groups = len(kernels) * ROTATIONS
        self.outs = [1] * (stages - 1) + [scale * scale]   # what a group answers, per stage
        reach = max(abs(v) for name in kernels for tap in KERNELS[name] for v in tap)
        self.halo = max(stages * reach, self.skip_size // 2)
        self.rows = CENTRE_LEVELS * BINS ** 3   # per kernel
        self.to_luma = frozen(nn.Conv2d(3, 1, 1, bias=False), LUMA)

        # the index table: each kernel's 256 centre values -> level * BINS³ + where its table
        # starts; then, for taps 1-3, the 511 differences -> bin * BINS^(3 - tap)
        level = torch.arange(256) * CENTRE_LEVELS // 256
        centre = torch.cat([level * BINS ** 3 + k * self.rows for k in range(len(kernels))])
        d = torch.arange(-255, 256)
        taps = torch.cat([difference_bin(d) * BINS ** (3 - t) for t in (1, 2, 3)])
        self.register_buffer("index", torch.cat([centre, taps]).to(torch.int32), persistent=False)
        offset = torch.zeros(G * 4, dtype=torch.int32)   # where each channel's part of the index table starts
        for g in range(G):
            offset[g * 4] = g // ROTATIONS * 256
            for t in (1, 2, 3):
                offset[g * 4 + t] = len(centre) + (t - 1) * 511
        self.register_buffer("offset", offset.view(1, G * 4, 1, 1), persistent=False)

        size, middle = 2 * reach + 1, (scale - 1) / 2
        self.reads, self.combines = nn.ModuleList(), nn.ModuleList()
        for out in self.outs:
            # read: channel g * 4 + t is the centre (t = 0) or tap t minus the centre, in 8-bit steps
            read = torch.zeros(G * 4, 1, size, size)
            for k, name in enumerate(kernels):
                for r in range(ROTATIONS):
                    g = k * ROTATIONS + r
                    (y0, x0), *others = [rotate(tap, r) for tap in KERNELS[name]]
                    read[g * 4, 0, reach + y0, reach + x0] = 255
                    for t, (y, x) in enumerate(others, start=1):
                        read[g * 4 + t, 0, reach + y, reach + x] += 255
                        read[g * 4 + t, 0, reach + y0, reach + x0] -= 255
            bias = torch.tensor([0.5, 255.5, 255.5, 255.5]).repeat(G)   # dropping the fraction rounds
            self.reads.append(frozen(nn.Conv2d(1, G * 4, size, padding=reach), read, bias))
            # combine: a group's answer o is the sub-pixel o turned by its rotation; rotations
            # are averaged and kernels added
            combine = torch.zeros(out, G * out)
            for g in range(G):
                for o in range(out):
                    dy, dx = rotate((o // scale - middle, o % scale - middle), g % ROTATIONS)
                    to = round(dy + middle) * scale + round(dx + middle) if out > 1 else 0
                    combine[to, g * out + o] = 1 / ROTATIONS
            self.combines.append(frozen(nn.Conv2d(G * out, out, 1, bias=False), combine))
        # the tables, all kernels of a stage end to end; untrained, every answer is 0: bicubic
        self.tables = nn.ParameterList(nn.Parameter(torch.zeros(len(kernels) * self.rows, out)) for out in self.outs)

    def address(self, stage, luma):
        """(N, 1, H, W) luma -> (N, groups, H, W) table rows."""
        n, _, h, w = luma.shape
        steps = self.reads[stage](luma).to(torch.int32) + self.offset
        parts = self.index.index_select(0, steps.flatten()).view(n, self.groups, 4, h, w)
        return parts[:, :, 0] + parts[:, :, 1] + parts[:, :, 2] + parts[:, :, 3]   # Adds: WebGPU has no integer ReduceSum

    def residual(self, x):
        luma = self.to_luma(x).clamp(0, 1)   # whatever comes in, every address stays inside its table
        for stage, table in enumerate(self.tables):
            row = self.address(stage, luma)
            n, G, h, w = row.shape
            answers = table.index_select(0, row.flatten()).view(n, G, h, w, -1).permute(0, 1, 4, 2, 3)
            correction = self.combines[stage](answers.reshape(n, -1, h, w))
            if stage < self.stages - 1:
                luma = (luma + correction).clamp(0, 1)
        return torch.cat([correction, correction, correction], dim=1)   # on every colour: Y moves


NEIGHBOURS = [(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)]


class PatternFilter(models.Upscaler):
    """Bicubic colour, and a luma correction by a filter looked up for the 3x3 neighbourhood's pattern."""

    def __init__(self, dead_zone=DEAD_ZONE, scale=2):
        super().__init__(scale, skip=5)
        self.dead_zone = dead_zone
        self.halo = max(1, self.skip_size // 2)
        self.to_luma = frozen(nn.Conv2d(3, 1, 1, bias=False), LUMA)
        # the one 3x3 convolution: neighbour minus centre, for each of the 8 neighbours
        weight = torch.zeros(8, 1, 3, 3)
        for i, (dy, dx) in enumerate(NEIGHBOURS):
            weight[i, 0, 1 + dy, 1 + dx] = 1
            weight[i, 0, 1, 1] = -1
        self.differences = frozen(nn.Conv2d(1, 8, 3, padding=1, bias=False), weight)
        # the 8 signs (-1, 0, 1) as base-3 digits, added up in two halves of four: each 0 .. 80
        halves = torch.zeros(2, 8)
        halves[0, :4] = halves[1, 4:] = torch.tensor([1.0, 3, 9, 27])
        self.to_halves = frozen(nn.Conv2d(8, 2, 1), halves, torch.tensor([40.0, 40.0]))
        # per pattern and sub-pixel: 8 weights and a bias; zero, so untrained it is bicubic
        self.filters = nn.Parameter(torch.zeros(3 ** 8, scale * scale * 9))

    def pattern(self, luma):
        """(N, 1, H, W) luma -> the 8 differences (N, 8, H, W), and each pixel's row (N, H, W)."""
        d = self.differences(luma)
        signs = torch.floor(d * (255 / (2 * self.dead_zone)) + 0.5).clamp(-1, 1)   # -1, 0 or 1 per neighbour; WebGPU has no Round
        low, high = self.to_halves(signs).unbind(1)
        return d, (low + 81 * high).to(torch.int32)                       # 0 .. 3^8 - 1, exact in float32; WebGPU has no integer Mul

    def residual(self, x):
        luma = self.to_luma(x).clamp(0, 1)
        d, row = self.pattern(luma)
        n, _, h, w = d.shape
        weights = self.filters.index_select(0, row.flatten()).view(n, h, w, -1, 9).permute(0, 3, 4, 1, 2)
        correction = (weights[:, :, :8] * d[:, None]).sum(dim=2) + weights[:, :, 8]   # 8 weights, then the bias
        return torch.cat([correction, correction, correction], dim=1)      # on every colour: Y moves


MODELS = {   # name -> (class, arguments): the cheapest and the best binned tables, and the 3x3 pattern filter
    "S": (FastLut, {"kernels": "S", "stages": 1}),
    "SDY x2": (FastLut, {"kernels": "SDY", "stages": 2}),
    "3x3 pattern": (PatternFilter, {"dead_zone": DEAD_ZONE}),
}


def build(name):
    """A fresh, untrained model."""
    model_class, arguments = MODELS[name]
    return model_class(scale=SCALE, **arguments).to(DEVICE)


def slug(name):
    return name.replace(" ", "")

In [3]:
"""Checks: untrained every model is bicubic, a binned table turns with its input, and every
address is the one worked out by hand."""

models.check_skip_is_bicubic(SCALE)
lr_probe, _ = training.val_screens(manifest, count=8)
with torch.no_grad(), training.full_precision():
    bicubic_out = trim(models.bicubic(lr_probe, SCALE))
    for name in MODELS:
        model = build(name)
        assert model.halo <= manifest["halo"], f"{name} reaches further than the patches' halo"
        gap = (trim(model(lr_probe)) - bicubic_out).abs().max().item()
        assert gap < 1e-5, f"untrained {name} is not bicubic ({gap:.1e})"
        turned = "-"
        if isinstance(model, FastLut):   # its rotations share one table, so it turns with its input
            for table in model.tables:
                table.normal_(0, 0.05)   # random answers, so a turn shows
            grey = torch.randint(0, 256, (1, 3, 40, 40), device=DEVICE) / 255
            gap_turned = (model(torch.rot90(grey, 1, (2, 3))) - torch.rot90(model(grey), 1, (2, 3))).abs().max().item()
            assert gap_turned < 1e-5, f"{name} does not turn with its input ({gap_turned:.1e})"
            turned = f"{gap_turned:.0e}"
        entries = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name:<12} halo {model.halo}, untrained against bicubic {gap:.0e}, turned {turned}; "
              f"{entries:,} entries: {entries * 2 / 1e6:.2f} MB in float16")

# the rows read on a grey 8-bit picture, against ones worked out by hand
grey = torch.randint(0, 256, (1, 1, 12, 12), device=DEVICE) / 255
pixel = (grey[0, 0] * 255).round().long().cpu()
binned = build("S").address(0, grey)[0].cpu()
_, patterns = build("3x3 pattern").pattern(grey)
patterns = patterns[0].cpu()
wrong = 0
for y in range(2, 10):
    for x in range(2, 10):
        for g in range(ROTATIONS):   # the binned table: centre level and binned differences, per rotation
            values = [pixel[y + dy, x + dx].item() for dy, dx in (rotate(tap, g) for tap in KERNELS["S"])]
            expected = values[0] * CENTRE_LEVELS // 256 * BINS ** 3
            for t in (1, 2, 3):
                expected += difference_bin(torch.tensor(values[t] - values[0])).item() * BINS ** (3 - t)
            wrong += int(binned[g, y, x].item() != expected)
        expected = 0                 # the pattern: each neighbour's sign beyond the dead zone, base 3
        for i, (dy, dx) in enumerate(NEIGHBOURS):
            difference = pixel[y + dy, x + dx].item() - pixel[y, x].item()
            sign = 0 if abs(difference) < DEAD_ZONE else 1 if difference > 0 else -1
            expected += (sign + 1) * 3 ** i
        wrong += int(patterns[y, x].item() != expected)
assert wrong == 0, f"{wrong} rows differ from the hand-worked ones"
print("every row is the one worked out by hand")

S            halo 2, untrained against bicubic 4e-07, turned 4e-07; 219,488 entries: 0.44 MB in float16
SDY x2       halo 4, untrained against bicubic 4e-07, turned 2e-07; 823,080 entries: 1.65 MB in float16
3x3 pattern  halo 2, untrained against bicubic 4e-07, turned -; 236,196 entries: 0.47 MB in float16
every row is the one worked out by hand


## 2. Train

`training.fit`, the shared recipe. What trains is the table itself: every pixel reads one entry, and the loss reaches
exactly the entries that were read. `S` is trained at three seeds, the spread luck alone makes, the others at one.

In [ ]:
def run_name(name, seed=0):
    return name + (f", seed {seed}" if seed else "")


runs = {}
for name in MODELS:
    for seed in SEEDS if name == "S" else SEEDS[:1]:
        torch.manual_seed(seed)
        model = build(name)
        history, best = training.fit(model, RUN_DIR / f"{slug(name)}_seed{seed}.pt", manifest, seed=seed, verbose=0)
        runs[run_name(name, seed)] = model.eval(), history
        print(f"{run_name(name, seed):<14} best epoch {best['epoch']:3} of {len(history):3}")

training.plot_history({name: history for name, (_, history) in runs.items()}, training.select_bicubic(manifest),
                      key="ssim", loss_name="SSIM-Y + L1 loss")

## 3. Scores

Every model on the validation split, beside notebook 10's fine-tuned interpolated tables and notebook 7's model at
three seeds. **better** / **worse** means a gap wider than the spread of `S`'s three seeds; **luck** means this set
cannot tell the two apart.

In [ ]:
scores = {}
for name, (model, _) in runs.items():
    scores[name] = training.val_scores(lambda lr, condition: model(lr), manifest)
    training.record(NOTEBOOK, f"binned {name}" if name.startswith(("S", "SDY")) else name, scores[name])

training.show(training.compare({"bicubic": bicubic, **others, **scores}, manifest, base=bicubic))

mean = lambda s: s["ssim"].mean()
seeds = [mean(scores[run_name("S", seed)]) for seed in SEEDS]
spread = np.ptp(seeds)
print(f"\nS at {len(SEEDS)} seeds: {spread:.4f} SSIM-Y apart")


def verdict(a, b, label):
    word = "luck" if abs(a - b) <= spread else ("better" if a > b else "worse")
    print(f"{label:<50}{a - b:+.4f} SSIM-Y  {word}")


best_name = max(MODELS, key=lambda name: mean(scores[name]))
verdict(float(np.mean(seeds)), mean(bicubic), "S (mean of its seeds) over bicubic")
for name in list(MODELS)[1:]:
    verdict(mean(scores[name]), float(np.mean(seeds)), f"{name} over S")
for name in ("S", "SDY x2"):
    verdict(mean(scores[name]), mean(others[f"LUT {name} table, fine-tuned"]),
            f"{name} over notebook 10's interpolated {name} table")
verdict(float(np.mean([mean(s) for n, s in others.items() if n.startswith("luma")])), mean(scores[best_name]),
        f"notebook 7's model (mean of its seeds) over {best_name}")

### What it looks like

Four validation screens at a slice QP under 26: bicubic, the best model here and the target. A binned table answers in
steps, so look for banding on smooth gradients as well as at the text.

In [ ]:
lr, hr = training.val_screens(manifest)
with torch.no_grad():
    training.show_patches({"bicubic": trim(models.bicubic(lr, SCALE)), best_name: trim(runs[best_name][0](lr)),
                           "target": hr})

## 4. Export for the browser

Every model, at the client's tile (a 320x180 step) and as a whole 1080p frame: each is a
tier. A graph reshapes its look-up at the size it was exported at, so the whole-frame graph is
exported at the frame's size.

**float32 only.** An address is only exact if every tap rounds to the right 8-bit step. In
float16 a value can round to the step beside it - and a second stage reads a corrected luma,
which is no 8-bit value at all - so a float16 graph reads the neighbouring row for about 2%
of `SDY x2`'s values. That is not a model worth shipping, so every model ships in float32 -
the pattern filter's float16 graph would pass, but its table is small anyway.

**The check allows for rounding.** Even in float32, a tap that lands exactly between two
steps can round one way in torch and the other in a graph, and read the neighbouring row: the
same kind of error as the binning itself. So a graph passes if fewer than 0.5% of its output
values differ from torch's by more than the usual tolerance.

In [ ]:
def check_and_time(info, module, shape, tolerance=2e-3, allowed=0.005):
    """Check a written graph against the torch module on an 8-bit picture - at most `allowed`
    of its values more than `tolerance` apart - then time it on the CPU and print one line."""
    picture = np.round(np.random.rand(1, 3, *shape) * 255).astype(np.float32) / 255
    with torch.no_grad():
        expected = copy.deepcopy(module).cpu()(torch.from_numpy(picture)).numpy()
    apart = np.abs(webexport.run(info["path"], picture) - expected) > tolerance
    assert apart.mean() <= allowed, f"{info['path'].name}: {apart.mean():.2%} of values apart"
    ms = webexport.time_graph(info["path"], shape)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms), f"  {apart.mean():.3%} of values apart")


(step_width, step_height), (frame_width, frame_height) = webexport.CLIENT_STEP, webexport.FRAME
for name in MODELS:
    tag = f"fastlut_{slug(name).lower()}"
    module = runs[name][0]
    halo = module.halo
    tile_shape = (step_height + 2 * halo, step_width + 2 * halo)
    frame_shape = (frame_height // SCALE + 2 * halo, frame_width // SCALE + 2 * halo)
    file = f"upscale_{tag}_tile{step_width}x{step_height}"
    check_and_time(webexport.export(module, f"{file}.onnx", tile_shape, label=f"{tag} fp32", halo=halo),
                   module, tile_shape)
    check_and_time(webexport.export(module, f"{file}_frame.onnx", frame_shape, label=f"{tag} fp32 · whole frame",
                                    halo=halo), module, frame_shape)
print("compare in a browser: uv run src/upscale/benchmark/main.py")

## 5. What it says

*(written after the run)*